# Tutorial 03 — Pretraining and decoding

[Lecture notes](../lectures/03_pretraining.md) · [Slides](../slides/03_pretraining.pptx) · [Exercise sheet](../exercises/03_pretraining.md)

**40 minutes, CPU, NumPy only.** You will train a character bigram model with a gradient you can check, watch held-out loss turn upward when the training set is small, and then decode: first from one fixed distribution over five tokens, then from the trained model. The corpus is four sentences written for this course. The model remembers one previous character and shows the training loop, not what a large model can do.

| Minutes | Step |
|---|---|
| 0–10 | Build training pairs, train, check the gradient |
| 10–15 | Halve the data and watch held-out loss |
| 15–22 | Temperature and top-k on one distribution |
| 22–30 | Greedy, beam search and a repetition penalty on the trained model |
| 30–40 | Your turn: two functions with tests |

In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Build document-bounded training pairs

The model has one row of next-character logits for each previous character. Each document gets its own start and end symbol, so no pair crosses from one document into the next.

Before running: the weights start near zero. What loss do you expect?

In [2]:
train_docs=["the class starts today", "the lab opens today", "the class studies models", "the lab studies language"]
dev_docs=["the class opens today", "the lab starts today"]
vocab=["<unk>","<bos>","<eos>"]+sorted(set("".join(train_docs)))
ids={c:i for i,c in enumerate(vocab)}
def pairs(docs):
    pairs=[]
    for doc in docs:
        s=[1]+[ids.get(c,0) for c in doc]+[2]
        pairs.extend(zip(s[:-1],s[1:]))
    return np.array(pairs).T
x,y=pairs(train_docs); dx,dy=pairs(dev_docs)
W=rng.normal(0,.01,(len(vocab),len(vocab)))
def softmax(z):
    z=z-z.max(-1,keepdims=True); e=np.exp(z); return e/e.sum(-1,keepdims=True)
def loss(W,x,y): return -np.log(softmax(W[x])[np.arange(len(y)),y]).mean()
print("vocabulary size",len(vocab),"training pairs",len(y))
print("initial dev loss",loss(W,dx,dy))

vocabulary size 22 training pairs 93
initial dev loss 3.095091471214213


The vocabulary has 22 symbols and ln 22 = 3.091, so the initial loss of 3.095 is the loss of a uniform guess. Any model you train should start near ln V.

## 2. Train with an explicit gradient

For one pair the gradient of the loss with respect to the logits is the predicted distribution minus the one-hot target. `np.add.at` adds up the gradients of pairs that share a previous character; plain indexed assignment would keep only one of them.

Predict what each row of the gradient sums to.

In [3]:
history=[]
initial=loss(W,x,y)
for step in range(301):
    p=softmax(W[x]); delta=p.copy(); delta[np.arange(len(y)),y]-=1
    grad=np.zeros_like(W); np.add.at(grad,x,delta/len(y))
    assert np.allclose(grad.sum(axis=1),0,atol=1e-12)
    W-=2.0*grad
    if step%50==0:
        row=(step,float(loss(W,x,y)),float(loss(W,dx,dy)))
        history.append(row); print("step/train/dev",row)
assert loss(W,x,y)<initial

step/train/dev (0, 3.045231012170305, 3.0446233298431484)
step/train/dev (50, 1.6905940634511638, 1.6635518592424507)
step/train/dev (100, 1.356801146817496, 1.3318752777315355)
step/train/dev (150, 1.22394333858768, 1.2022335226263805)
step/train/dev (200, 1.1564515259084982, 1.1372884440034778)
step/train/dev (250, 1.1177126100133161, 1.1003185146807442)
step/train/dev (300, 1.0933783461892714, 1.077142523787638)


Each gradient row sums to zero, because a probability vector and a one-hot vector both sum to one. Training and development loss fall together to about 1.09 and 1.08. The two development sentences reuse the words of the training sentences, so this development set cannot show overfitting. Section 4 uses one that can.

## 3. Check the gradient, then sample

A finite-difference check compares the derived gradient with a numerical slope. Then sample one string from the trained model at temperature 1.

In [4]:
p=softmax(W[x]); p[np.arange(len(y)),y]-=1
G=np.zeros_like(W); np.add.at(G,x,p/len(y))
i,j=int(x[0]),int(y[0]); eps=1e-5
plus=W.copy(); minus=W.copy(); plus[i,j]+=eps; minus[i,j]-=eps
fd=(loss(plus,x,y)-loss(minus,x,y))/(2*eps)
assert np.isclose(fd,G[i,j],atol=1e-7)
cur=1; output=[]
for _ in range(100):
    cur=int(rng.choice(len(vocab),p=softmax(W[cur])))
    if cur==2: break
    output.append(vocab[cur])
print("sample:","".join(output))
print("FLOP estimate for 100M parameters / 2B tokens:",6*100_000_000*2_000_000_000)

sample: tmodie clage la
FLOP estimate for 100M parameters / 2B tokens: 1200000000000000000


The derivation passes the check. The sample has English letter pairs and no English words, which is all a model with one character of memory can do. The last line is the 6ND estimate from the lecture: 1.2 × 10¹⁸ floating-point operations.

## 4. Less data: held-out loss turns upward

Train a fresh model on the first two sentences only, and hold out the other two. The held-out sentences now contain character pairs the model never saw.

Predict the shape of the held-out curve.

In [5]:
def gradient(W, x, y):
    p = softmax(W[x]); p[np.arange(len(y)), y] -= 1
    G = np.zeros_like(W); np.add.at(G, x, p / len(y))
    return G

xs, ys = pairs(train_docs[:2]); hx, hy = pairs(train_docs[2:])
W_init = rng.normal(0, .01, W.shape)
W_small = W_init.copy()
curve = []
for step in range(401):
    if step % 40 == 0:
        curve.append((step, float(loss(W_small, xs, ys)), float(loss(W_small, hx, hy))))
        print(f"step {step:3d}  train {curve[-1][1]:.3f}  held-out {curve[-1][2]:.3f}")
    W_small -= 2.0 * gradient(W_small, xs, ys)
assert curve[-1][1] < curve[1][1] and curve[-1][2] > min(c[2] for c in curve)

step   0  train 3.094  held-out 3.092
step  40  train 1.448  held-out 2.415
step  80  train 1.062  held-out 2.417
step 120  train 0.935  held-out 2.491
step 160  train 0.879  held-out 2.564
step 200  train 0.848  held-out 2.628
step 240  train 0.830  held-out 2.682
step 280  train 0.818  held-out 2.730
step 320  train 0.809  held-out 2.771
step 360  train 0.802  held-out 2.807
step 400  train 0.797  held-out 2.840


Training loss keeps falling, from 1.448 at step 40 to 0.797 at step 400. Held-out loss reaches its minimum near step 40 to 80 and then rises to 2.840, because the model grows more certain about pairs that the held-out text does not follow. The weights at the last step are worse on held-out text than weights you had earlier and threw away. Exercise B in section 7 keeps them.

## 5. Decoding: one distribution, several ways to reshape it

A model has read "Darasa linaanza" (Swahili: "the class starts") and gives five candidate next words these probabilities: *leo* (today) 0.50, *kesho* (tomorrow) 0.25, *sasa* (now) 0.15, *hapa* (here) 0.07, *tena* (again) 0.03. The numbers are invented for the example.

Temperature T divides the logits by T before the softmax. Top-k keeps the k most probable tokens and renormalises. Work out the T = 0.5 row on paper before running.

In [6]:
words = ["leo", "kesho", "sasa", "hapa", "tena"]
base = np.array([0.50, 0.25, 0.15, 0.07, 0.03])

def with_temperature(probs, T):
    return softmax(np.log(probs) / T)

def top_k_filter(probs, k):
    keep = np.argsort(-probs)[:k]
    out = np.zeros_like(probs); out[keep] = probs[keep]
    return out / out.sum()

def entropy_bits(probs):
    nz = probs[probs > 0]
    return float(-(nz * np.log2(nz)).sum())

def show(name, probs):
    print(f"{name:20s}" + "".join(f"{v:7.3f}" for v in probs) + f"   {entropy_bits(probs):5.2f}")

print(" " * 20 + "".join(f"{w:>7s}" for w in words) + "   entropy (bits)")
show("model, T = 1", base)
show("T = 0.5", with_temperature(base, 0.5))
show("T = 2", with_temperature(base, 2.0))
show("top-k, k = 2", top_k_filter(base, 2))
print("greedy picks:", words[int(base.argmax())])
assert np.allclose(with_temperature(base, 1.0), base)
assert np.isclose(with_temperature(base, 0.5)[0], 0.25 / (base ** 2).sum())

                        leo  kesho   sasa   hapa   tena   entropy (bits)
model, T = 1          0.500  0.250  0.150  0.070  0.030    1.83
T = 0.5               0.734  0.183  0.066  0.014  0.003    1.15
T = 2                 0.348  0.246  0.191  0.130  0.085    2.17
top-k, k = 2          0.667  0.333  0.000  0.000  0.000    0.92
greedy picks: leo


T = 0.5 squares each probability and renormalises: *leo* rises from 0.50 to 0.73 and the entropy falls from 1.83 to 1.15 bits. T = 2 takes square roots and flattens the distribution: *tena* rises from 0.03 to 0.085. Top-k with k = 2 removes three words whatever their probability. None of these changes the model. Each changes what you sample from it, so the decoding settings belong in your experiment's configuration next to the seed.

## 6. Decoding a sequence: greedy, beam search, repetition

Now decode from the bigram model trained in section 2. Greedy takes the most probable next character each time. Beam search keeps the 3 best partial strings at every step and returns the finished string with the highest total log-probability. The repetition penalty divides the logit of every character already produced by 1.3 before each choice.

Predict what greedy decoding prints.

In [7]:
P = softmax(W)

def greedy(penalty=1.0, max_len=40):
    cur, out, logp = 1, [], 0.0
    for _ in range(max_len):
        z = W[cur].copy()
        seen = [ids[c] for c in set(out)]
        z[seen] = np.where(z[seen] > 0, z[seen] / penalty, z[seen] * penalty)
        nxt = int(z.argmax()); logp += float(np.log(P[cur, nxt])); cur = nxt
        if cur == 2: break
        out.append(vocab[cur])
    return "".join(out), logp

def beam_search(width=3, max_len=40):
    beams, finished = [(0.0, [1])], []
    for _ in range(max_len):
        candidates = [(lp + float(np.log(P[seq[-1], j])), seq + [j]) for lp, seq in beams for j in range(len(vocab))]
        candidates.sort(key=lambda c: -c[0])
        beams = []
        for lp, seq in candidates[:width]:
            (finished if seq[-1] == 2 else beams).append((lp, seq))
        if not beams: break
    lp, seq = max(finished, key=lambda c: c[0])
    return "".join(vocab[t] for t in seq[1:-1]), lp

for name, (text, logp) in [("greedy", greedy()), ("greedy, penalty 1.3", greedy(1.3)), ("beam, width 3", beam_search())]:
    print(f"{name:20s} log-prob {logp:8.2f}  {text!r}")
assert beam_search()[1] > greedy()[1]

greedy               log-prob   -31.76  'the lab lab lab lab lab lab lab lab lab '
greedy, penalty 1.3  log-prob   -11.21  'the lab s clay'
beam, width 3        log-prob    -5.20  'the lay'


Greedy decoding falls into a loop and never reaches the end symbol in 40 characters: after "lab " the most probable character leads back to "lab". The penalty breaks the loop and gives "the lab s clay", which is no more sensible. Beam search finds a much higher log-probability, −5.20, with "the lay", a string that is in none of the sentences. It prefers short outputs, because every extra character multiplies in another probability below one. High probability under the model and good text are different things. Which of the three would you report as "the model's output"?

## 7. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. Nucleus (top-p) filtering.** Sort the tokens from most to least probable. Keep the smallest set whose probabilities add up to at least `p`. Set the rest to zero and renormalise. The input need not be sorted, and the output must be in the input's order.

In [ ]:
# YOUR TURN
def top_p_filter(probs, p):
    """Keep the smallest set of most probable tokens with total mass >= p; renormalise."""
    raise NotImplementedError("your turn")

In [9]:
assert np.allclose(top_p_filter(base, 0.8), [0.5 / 0.9, 0.25 / 0.9, 0.15 / 0.9, 0, 0])
assert np.allclose(top_p_filter(base, 0.5), [1, 0, 0, 0, 0])
assert np.allclose(top_p_filter(base, 1.0), base)
shuffled = base[[3, 0, 4, 2, 1]]
assert np.allclose(top_p_filter(shuffled, 0.8), [0, 0.5 / 0.9, 0, 0.15 / 0.9, 0.25 / 0.9])
flat = np.full(5, 0.2)
print("tokens kept at p = 0.8:  peaked distribution", int((top_p_filter(base, 0.8) > 0).sum()), " flat distribution", int((top_p_filter(flat, 0.8) > 0).sum()))
print(" " * 20 + "".join(f"{w:>7s}" for w in words) + "   entropy (bits)")
show("model, T = 1", base)
show("top-p, p = 0.8", top_p_filter(base, 0.8))
show("T = 2, then p = 0.8", top_p_filter(with_temperature(base, 2.0), 0.8))
print("A passed")

tokens kept at p = 0.8:  peaked distribution 3  flat distribution 4
                        leo  kesho   sasa   hapa   tena   entropy (bits)
model, T = 1          0.500  0.250  0.150  0.070  0.030    1.83
top-p, p = 0.8        0.556  0.278  0.167  0.000  0.000    1.42
T = 2, then p = 0.8   0.380  0.269  0.208  0.142  0.000    1.91
A passed


Top-k keeps the same number of tokens everywhere. Top-p keeps 3 tokens of the peaked distribution and 4 of the flat one: the cut follows the model's own confidence.

**B. Keep the best checkpoint.** Train a copy of `W0` by gradient descent with `gradient` from section 4. Every `every` steps, starting at step 0 and before the update, measure the held-out loss. Return the weights with the lowest held-out loss, the step at which they were measured, and that loss.

In [ ]:
# YOUR TURN
def train_with_selection(W0, x, y, held_x, held_y, lr, steps, every):
    """Return (best_W, best_step, best_loss): the checkpoint with the lowest held-out loss."""
    raise NotImplementedError("your turn")

In [11]:
before = W_init.copy()
best_W, best_step, best_loss = train_with_selection(W_init, xs, ys, hx, hy, lr=2.0, steps=400, every=20)
assert np.array_equal(W_init, before)                                # the starting weights are not changed
assert np.isclose(loss(best_W, hx, hy), best_loss)                   # the returned weights are the best ones, not the last
assert 0 < best_step < 400 and best_loss < curve[-1][2]
for lr in [0.5, 2.0, 8.0]:
    _, s, l = train_with_selection(W_init, xs, ys, hx, hy, lr=lr, steps=400, every=20)
    print(f"learning rate {lr:3.1f}: best held-out loss {l:.3f} at step {s}")
print("B passed")

learning rate 0.5: best held-out loss 2.394 at step 220
learning rate 2.0: best held-out loss 2.395 at step 60
learning rate 8.0: best held-out loss 2.420 at step 20
B passed


The three learning rates reach almost the same best held-out loss at very different steps. A fixed number of steps would have favoured one of them. Selecting the checkpoint on held-out data makes the comparison fair, and it also means the held-out loss is now a selection score: report the final number on a test set that played no part.

## Discussion

1. Training perplexity keeps improving while the system gets worse at its task. Give two different reasons.
2. Some test documents also appear in the training data of the larger of two models. What happens to the comparison?
3. Two teams report results for the same model on the same test set, one with greedy decoding and one with temperature 1.0. Can their numbers be compared?

## From the toy to your task

Find the decoding settings in your task's baseline code: temperature, top-p, top-k, beam width, maximum length, repetition penalty. Tasks 6 and 7 score generated text, so these settings change your score. The exercise sheet asks you to record them.

A PyTorch run of a small causal Transformer on the same corpus is in `extensions/train_tiny_decoder.py`, with a [measured 100-step run](../validation/tiny_decoder_run.json).

Submit this notebook with both tests passing and a 150–250-word note: the checkpoint you selected and why, and which decoding method you would use for your task's baseline.